# TP 2 — Préparation des données et forêt aléatoire (cancer du sein bruité)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/ai_for_medicine/blob/main/ia-medecine/notebooks/iam-s2-tp.ipynb)

## Objectif

En séance 1, le jeu de données était **propre**. Les données médicales
réelles le sont rarement : elles contiennent des **valeurs manquantes**,
des **valeurs aberrantes** et des **variables catégorielles** qu'il faut
préparer.

Dans ce TP :

1. on travaille sur une version **bruitée** du jeu de données du cancer du
   sein (valeurs manquantes, valeurs aberrantes, variables catégorielles) ;
2. on applique la **préparation des données** : imputation → valeurs
   aberrantes → encodage → mise à l'échelle ;
3. on entraîne une **forêt aléatoire** et on la compare à la régression
   logistique.

La méthode reste celle de la séance 1 : **explorer → visualiser → préparer
→ entraîner → évaluer → valider**. Cette séance porte sur l'étape
**préparer**, sautée en séance 1 parce que les données étaient propres.

**Comment travailler** : Fichier → *Enregistrer une copie dans Drive* ;
exécutez les cellules dans l'ordre (Maj + Entrée) ; complétez les cellules
marquées `# TODO` en remplaçant les `...`.

## 1. Mise en place

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, classification_report

## 2. Charger et découvrir les données

En séance 1, le jeu de données *Breast Cancer Wisconsin* était **parfaitement
propre** : aucune valeur manquante, aucune valeur aberrante, que des
variables numériques. C'est rare en pratique.

Vous recevez cette fois une version plus proche de **vraies données
cliniques** :

- des valeurs **manquent** (panne d'appareil, dossier incomplet) ;
- des mesures sont **aberrantes** (erreur de saisie, cas extrême) ;
- des **variables catégorielles** ont été ajoutées (tranche d'âge, statut
  ménopausique, localisation de la tumeur).

Premier travail : **repérer et comprendre** ces défauts avant tout modèle.

### Exercice 1.1 — Charger le jeu de données

Chargez le fichier CSV, puis définissez les noms des classes et les listes
de colonnes numériques et catégorielles.

**Pourquoi :** on commence toujours par regarder ses données.

*Indice :*
```python
df = pd.read_csv('https://raw.githubusercontent.com/racousin/ai_for_medicine/main/ia-medecine/donnees/breast_cancer_noisy.csv')
noms_classes = np.array(['maligne', 'bénigne'])   # cible : 0 = maligne, 1 = bénigne
colonnes_num = df.select_dtypes(include='number').columns.drop('target').tolist()
colonnes_cat = df.select_dtypes(exclude='number').columns.tolist()
```

In [ ]:
# TODO : chargez les données et définissez les trois variables
df = ...
noms_classes = ...
colonnes_num = ...
colonnes_cat = ...

### Exercice 1.2 — Inspecter la qualité des données

Utilisez `.info()` pour voir les types et les valeurs manquantes, et
`.describe()` pour vérifier les plages de valeurs.

**Pourquoi :** repérer les trois types de problèmes : valeurs manquantes
(`NaN`), valeurs aberrantes (extrêmes) et variables non numériques.

**Question :** combien de valeurs manquent ? Quelles variables ont des
valeurs suspectes ? Quelles colonnes sont catégorielles ?

In [ ]:
# TODO : df.info() puis df.describe()
...

*Votre réponse ici*

## 3. Visualiser les défauts

Avant de corriger, il faut **voir** les problèmes.

### Exercice 2.1 — Carte des valeurs manquantes

Visualisez où se trouvent les valeurs manquantes.

**Pourquoi :** voir si elles sont dispersées au hasard ou concentrées sur
certaines variables.

*Indice :* `sns.heatmap(df[colonnes_num].isnull(), cbar=True, yticklabels=False)`
ou `df[colonnes_num].isnull().sum().plot(kind='bar')`

In [ ]:
# TODO : carte des valeurs manquantes et/ou nombre par variable
...

### Exercice 2.2 — Boîtes à moustaches pour repérer les valeurs aberrantes

Tracez une boîte à moustaches de `mean radius` et une de `mean area`.

**Pourquoi :** une boîte à moustaches montre la médiane, les quartiles et
les points au-delà de 1,5 × IQR. Les valeurs extrêmes sautent aux yeux.

*Indice :* `fig, axes = plt.subplots(1, 2, figsize=(12, 5))`,
`sns.boxplot(y=df['mean radius'], ax=axes[0])`

In [ ]:
# TODO : deux boîtes à moustaches côte à côte
...

### Exercice 2.3 — Répartition des variables catégorielles

Tracez un diagramme en barres pour `age_group`, `menopause_status` et
`tumor_location`.

**Pourquoi :** connaître la répartition des modalités et repérer une
modalité très rare.

*Indice :* `df['age_group'].value_counts().plot(kind='bar')`

In [ ]:
# TODO : un diagramme en barres par variable catégorielle
...

## 4. Traiter les valeurs manquantes

La plupart des modèles refusent les `NaN` : il faut les traiter avant
l'entraînement.

| Stratégie | Quand l'utiliser |
|---|---|
| **Supprimer les lignes** | très peu de valeurs manquantes, beaucoup de données |
| **Imputer par la moyenne ou la médiane** | variables numériques |
| **Imputer par la modalité la plus fréquente** | variables catégorielles |
| **Imputer par un modèle** | cas complexes (avancé) |

On utilise `SimpleImputer` de scikit-learn : la **médiane** pour les
variables numériques (robuste aux valeurs aberrantes), la **modalité la
plus fréquente** pour les variables catégorielles.

### Exercice 3.1 — Compter les valeurs manquantes

Comptez les valeurs manquantes par colonne, puis au total.

*Indice :* `df.isnull().sum()`, `df.isnull().sum().sum()`

In [ ]:
# TODO : nombre de valeurs manquantes par colonne et au total
...

### Exercice 3.2 — Imputer les valeurs manquantes

Utilisez `SimpleImputer` avec `strategy='median'` pour les variables
numériques et `strategy='most_frequent'` pour les catégorielles.

**Pourquoi :** la médiane résiste mieux que la moyenne aux valeurs
aberrantes, et nos données en contiennent.

*Indice :*
```python
imputeur_num = SimpleImputer(strategy='median')
df[colonnes_num] = imputeur_num.fit_transform(df[colonnes_num])
```

La cellule suivante vérifie qu'il ne reste aucune valeur manquante.

In [ ]:
# TODO : imputez les colonnes numériques (médiane) et catégorielles
# (modalité la plus fréquente)
...

In [ ]:
assert df.isnull().sum().sum() == 0, "il reste des valeurs manquantes"
print("✅ Plus aucune valeur manquante")

## 5. Traiter les valeurs aberrantes

Les valeurs extrêmes peuvent fausser l'entraînement. On les repère et on
les **écrête** avec la **méthode de l'IQR** :

- **IQR** = Q3 − Q1 (écart interquartile) ;
- **borne basse** = Q1 − 1,5 × IQR ;
- **borne haute** = Q3 + 1,5 × IQR ;
- une valeur hors des bornes est remplacée par la borne la plus proche.

**Pourquoi écrêter plutôt que supprimer ?** En médecine, on ne veut pas
perdre de patients : l'écrêtage garde le patient tout en réduisant
l'influence de la valeur extrême.

### Exercice 4.1 — Repérer et écrêter les valeurs aberrantes

Pour chaque variable numérique, écrêtez les valeurs hors des bornes IQR.
Affichez le minimum et le maximum de `mean radius` et `mean area` avant et
après.

*Indice :*
```python
for col in colonnes_num:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    df[col] = df[col].clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)
```

In [ ]:
print(f"Avant : mean radius max {df['mean radius'].max():.1f}, "
      f"mean area max {df['mean area'].max():.1f}")

# TODO : écrêtez chaque colonne numérique aux bornes IQR
...

print(f"Après : mean radius max {df['mean radius'].max():.1f}, "
      f"mean area max {df['mean area'].max():.1f}")

### Exercice 4.2 — Visualiser après l'écrêtage

Refaites les boîtes à moustaches de `mean radius` et `mean area` et
comparez avec l'exercice 2.2.

In [ ]:
# TODO : mêmes boîtes à moustaches qu'à l'exercice 2.2
...

## 6. Encoder les variables catégorielles

Les modèles calculent avec des **nombres**, pas avec du texte.

**Encodage one-hot** : pour une variable sans ordre naturel, on crée **une
colonne 0/1 par modalité**.

| tumor_location | → | upper_inner | central | lower_inner |
|---|---|---|---|---|
| upper_inner | | 1 | 0 | 0 |
| central | | 0 | 1 | 0 |
| lower_inner | | 0 | 0 | 1 |

**Pourquoi ne pas numéroter (upper_inner = 0, central = 1, lower_inner = 2) ?**
Le modèle y lirait un **ordre** (upper_inner < central < lower_inner) qui
n'a pas de sens ici.

### Exercice 5.1 — Encodage one-hot

Utilisez `pd.get_dummies()` pour encoder les colonnes catégorielles.

*Indice :* `df = pd.get_dummies(df, columns=colonnes_cat, drop_first=True)`.
`drop_first=True` retire une colonne redondante par variable : si toutes
les autres valent 0, la modalité retirée est connue. (scikit-learn propose
aussi `OneHotEncoder`, utilisé dans le défi.)

**Question :** combien de variables explicatives a-t-on après l'encodage ?

In [ ]:
# TODO : encodez colonnes_cat
df = ...
df.head()

*Votre réponse ici*

## 7. Mise à l'échelle

Comme en séance 1, on standardise les variables (moyenne 0, écart-type 1),
ce qui aide la régression logistique :

$$x_{\text{standardisé}} = \frac{x - \mu}{\sigma}$$

**Rappel :** `fit` sur l'apprentissage, `transform` sur l'apprentissage et
sur le test.

> **Remarque.** Pour rester simple, ce TP a imputé et écrêté **tout** le
> tableau avant de séparer apprentissage et test : les médianes et les
> bornes ont donc vu les patients du test, une petite **fuite
> d'information**. La bonne méthode (séparer d'abord, puis ajuster chaque
> étape sur l'apprentissage seul) est celle du défi.

### Exercice 6.1 — Séparer et mettre à l'échelle

Séparez les variables (`X`) de la cible (`y`), coupez en apprentissage et
test (80/20, `random_state=42`), puis standardisez avec `StandardScaler`.

*Indice :* comme en séance 1 ; nommez les résultats `X_app_s` et `X_test_s`.

In [ ]:
X = df.drop(columns='target')
y = df['target']

# TODO : séparation 80/20 puis standardisation
X_app, X_test, y_app, y_test = ...
X_app_s = ...
X_test_s = ...

## 8. Modèle de référence : la régression logistique

Avant d'essayer un nouveau modèle, on établit toujours une **référence**
avec un modèle connu : elle dit si le nouveau modèle fait vraiment mieux.

### Exercice 7.1 — Régression logistique de référence

Entraînez une `LogisticRegression(max_iter=5000)` sur les données
standardisées. Affichez l'exactitude et le `classification_report`.

In [ ]:
# TODO : entraînez lr_model, prédisez y_pred_lr sur X_test_s
lr_model = ...
y_pred_lr = ...

## 9. La forêt aléatoire

Une **forêt aléatoire** est un **ensemble** de nombreux **arbres de
décision**, chacun entraîné sur un tirage aléatoire des patients et des
variables. La prédiction finale est le **vote majoritaire** des arbres.

```
             Données
          /    |     \
   Arbre 1  Arbre 2  ...  Arbre N
      |        |            |
   préd. 1  préd. 2  ...  préd. N
          \    |     /
        vote majoritaire
      → prédiction finale
```

| Propriété | Régression logistique | Forêt aléatoire |
|---|---|---|
| **Frontière de décision** | linéaire | non linéaire |
| **Mise à l'échelle nécessaire** | oui | non |
| **Valeurs aberrantes** | sensible | robuste |
| **Interprétation** | coefficients | importances des variables |
| **Risque de surapprentissage** | faible | moyen (réglable) |

**Idée clé :** chaque arbre voit un tirage différent des données, donc fait
des **erreurs différentes** ; le vote les compense. Comme plusieurs avis
médicaux : le consensus est souvent plus fiable que chaque avis isolé.

### Exercice 8.1 — Entraîner une forêt aléatoire

Entraînez un `RandomForestClassifier(n_estimators=100, random_state=42)` et
comparez son exactitude à celle de la régression logistique.

**Remarque :** la forêt n'a pas besoin de données standardisées, mais on
garde les mêmes données pour une comparaison équitable.

In [ ]:
# TODO : entraînez rf_model, prédisez y_pred_rf sur X_test_s
rf_model = ...
y_pred_rf = ...

### Exercice 8.2 — Comparer les matrices de confusion

Affichez côte à côte les matrices de confusion des deux modèles.

**Pourquoi :** voir si les deux modèles font les mêmes types d'erreurs.

*Indice :* `fig, axes = plt.subplots(1, 2, figsize=(12, 5))`,
`ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr, display_labels=noms_classes, ax=axes[0])`

In [ ]:
# TODO : deux matrices de confusion côte à côte
...

### Exercice 8.3 — Importances des variables

Tracez les 15 variables les plus importantes selon la forêt.

**Pourquoi :** savoir quelles mesures guident le diagnostic du modèle.

*Indice :*
```python
importances = rf_model.feature_importances_
indices = np.argsort(importances)[-15:]
plt.barh(range(len(indices)), importances[indices])
plt.yticks(range(len(indices)), X.columns[indices])
```

**Question :** quelles variables sont les plus importantes ? Est-ce
cohérent avec les corrélations vues en séance 1 ?

In [ ]:
# TODO : diagramme en barres horizontales des 15 plus grandes importances
...

*Votre réponse ici*

## 10. Régler les hyperparamètres

Les **hyperparamètres** sont des réglages choisis **avant**
l'entraînement : le modèle ne les apprend pas des données.

| Hyperparamètre | Ce qu'il règle | Trop petit | Trop grand |
|---|---|---|---|
| `n_estimators` | nombre d'arbres | prédictions instables | lent, gain négligeable |
| `max_depth` | profondeur maximale d'un arbre | sous-apprentissage | surapprentissage |
| `min_samples_split` | patients minimum pour couper un nœud | surapprentissage | sous-apprentissage |

### Exercice 9.0 — Essayer des hyperparamètres

Entraînez quelques `RandomForestClassifier(n_estimators=..., max_depth=...,
min_samples_split=..., random_state=42)` avec des valeurs de votre choix et
comparez leur exactitude **sur l'apprentissage** et **sur le test**.

In [ ]:
# TODO : essayez plusieurs réglages et affichez les deux exactitudes
...

Choisir le réglage d'après le score **de test** revient à régler le modèle
sur le test. `GridSearchCV` compare les réglages par **validation croisée**
sur l'apprentissage seul, et laisse le test intact.

### Exercice 9.1 — Recherche en grille

`GridSearchCV` essaie toutes les combinaisons et garde la meilleure en
validation croisée :

```
n_estimators : [50, 100, 200] × max_depth : [3, 5, 10, None] × min_samples_split : [2, 5]
= 3 × 4 × 2 = 24 combinaisons, chacune en validation croisée à 5 plis = 120 entraînements
```

Lancez la recherche et affichez les meilleurs hyperparamètres et le meilleur
score.

*Indice :*
```python
grille = {'n_estimators': [50, 100, 200], 'max_depth': [3, 5, 10, None],
          'min_samples_split': [2, 5]}
recherche = GridSearchCV(RandomForestClassifier(random_state=42), grille,
                         cv=5, scoring='accuracy', n_jobs=-1)
recherche.fit(X_app_s, y_app)
```

In [ ]:
# TODO : définissez grille et recherche, puis lancez la recherche
grille = ...
recherche = ...

### Exercice 9.2 — Évaluer le meilleur modèle sur le test

Prédisez le test avec le meilleur modèle et affichez l'exactitude et le
`classification_report`.

*Indice :* `recherche.best_estimator_` est le meilleur modèle, déjà
réentraîné sur tout l'apprentissage.

**Question :** le réglage a-t-il amélioré la forêt par défaut ?

In [ ]:
# TODO : prédictions y_pred_best du meilleur modèle
y_pred_best = ...

*Votre réponse ici*

## 11. Validation croisée et comparaison finale

### Exercice 10.1 — Comparer les deux modèles en validation croisée

Lancez une validation croisée à 5 plis pour la régression logistique et
pour la meilleure forêt, puis affichez un tableau comparatif.

La standardisation est mise dans un **pipeline** avec la régression
logistique : elle est ainsi réajustée sur les 4 plis d'entraînement à
chaque tour, sans voir le pli de validation.

*Indice :*
```python
lr_scores = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
                            X, y, cv=5, scoring='accuracy')
rf_scores = cross_val_score(recherche.best_estimator_, X, y, cv=5, scoring='accuracy')
```

In [ ]:
# TODO : lr_scores et rf_scores, puis le tableau
lr_scores = ...
rf_scores = ...

## 12. Résumé

| Étape | Technique | Pourquoi |
|---|---|---|
| **Valeurs manquantes** | `SimpleImputer` (médiane) | combler les `NaN` sans perdre de patients |
| **Valeurs aberrantes** | écrêtage par l'IQR | limiter les extrêmes sans perdre de patients |
| **Variables catégorielles** | encodage one-hot (`get_dummies`) | transformer le texte en nombres |
| **Mise à l'échelle** | `StandardScaler` | mettre les variables sur la même échelle |
| **Référence** | régression logistique | le score à battre |
| **Nouveau modèle** | forêt aléatoire | un ensemble d'arbres de décision |
| **Réglage** | `GridSearchCV` | choisir les hyperparamètres en validation croisée |

```
Données brutes → manquantes → aberrantes → encodage → mise à l'échelle → modèle
```

**À retenir :** la préparation des données compte souvent plus que le choix
du modèle. Un modèle simple sur des données bien préparées bat souvent un
modèle complexe sur des données sales.

**Séance 3 :** les réseaux de neurones (`MLPClassifier`), un nouveau jeu de
données (maladie cardiaque) et la comparaison des trois modèles.